In [ ]:
from pyspark.sql import SparkSession

def get_spark_session(app_name="Calcular Violaciones Sanitarias RED"):
    """
    Crea o reutiliza una SparkSession existente.
    
    :param app_name: Nombre de la aplicación Spark.
    :return: Instancia activa de SparkSession.
    """
    return SparkSession.builder.appName(app_name).getOrCreate()

def calculate_red_violations(data_source_path: str, spark: SparkSession = None):
    """
    Procesa datos de inspección y muestra los 10 restaurantes con más violaciones del tipo RED.

    :param data_source_path: Ruta al archivo CSV (por ejemplo, en S3: 's3://bucket/nombre.csv').
    :param spark: Sesión Spark (si no se proporciona, se crea una nueva).
    :return: DataFrame con el resultado de la consulta.
    """
    if spark is None:
        spark = get_spark_session()

    print("Leyendo datos desde:", data_source_path)
    df = spark.read.option("header", "true").csv(data_source_path)

    # Crear vista temporal para consultas SQL
    df.createOrReplaceTempView("restaurant_violations")

    # Ejecutar consulta SQL para encontrar los top 10
    print("Ejecutando consulta SQL para encontrar violaciones RED...")
    result_df = spark.sql("""
        SELECT name, COUNT(*) AS total_red_violations
        FROM restaurant_violations
        WHERE violation_type = 'RED'
        GROUP BY name
        ORDER BY total_red_violations DESC
        LIMIT 10
    """)

    # Mostrar resultados
    print("Resultado:")
    result_df.show(truncate=False)

    return result_df

In [ ]:
# Ruta S3 al archivo CSV
data_source = "s3://seccion-b-uni-2026/food_establishment_data.csv"

# Crear o reutilizar sesión Spark
spark = get_spark_session()

# Ejecutar análisis
result_df = calculate_red_violations(data_source, spark)


In [ ]:
df_all = spark.read.csv(data_source, header=True, inferSchema=True)

In [ ]:
df_all.groupBy("violation_type").count().show()

In [ ]:
df_all.dtypes

In [ ]:
df_all = df_all.dropna()

In [ ]:
df_all.groupBy("violation_type").count().show()

In [1]:
# Iniciar sesión
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Food Inspection ML") \
    .getOrCreate()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
1,application_1777327681003_0002,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
# leer datos
# Ruta S3 al archivo CSV
data_source = "s3://seccion-b-uni-2026/food_establishment_data.csv"
df = spark.read.csv(data_source, 
                    header=True, inferSchema=True)

df.show(5)
df.printSchema()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-----------+-----------------+--------------------------+--------------+----------------+
|       name|inspection_result|inspection_closed_business|violation_type|violation_points|
+-----------+-----------------+--------------------------+--------------+----------------+
|100 LB CLAM|       Incomplete|                     false|          NULL|               0|
|100 LB CLAM|   Unsatisfactory|                     false|          BLUE|               5|
|100 LB CLAM|   Unsatisfactory|                     false|           RED|               5|
|100 LB CLAM|   Unsatisfactory|                     false|           RED|              10|
|100 LB CLAM|   Unsatisfactory|                     false|           RED|               5|
+-----------+-----------------+--------------------------+--------------+----------------+
only showing top 5 rows

root
 |-- name: string (nullable = true)
 |-- inspection_result: string (nullable = true)
 |-- inspection_closed_business: boolean (nullable = true)
 |-- vi

In [4]:
#  limpiando
df = df.fillna({
    "inspection_result": "UNKNOWN",
    "violation_type": "UNKNOWN"
})

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [5]:
# verificando
df.select("inspection_result", "violation_type").show(5)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-----------------+--------------+
|inspection_result|violation_type|
+-----------------+--------------+
|       Incomplete|       UNKNOWN|
|   Unsatisfactory|          BLUE|
|   Unsatisfactory|           RED|
|   Unsatisfactory|           RED|
|   Unsatisfactory|           RED|
+-----------------+--------------+
only showing top 5 rows

In [6]:
from pyspark.sql.functions import col

df.select([
    col(c).isNull().alias(c) for c in df.columns
]).show(5)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-----+-----------------+--------------------------+--------------+----------------+
| name|inspection_result|inspection_closed_business|violation_type|violation_points|
+-----+-----------------+--------------------------+--------------+----------------+
|false|            false|                     false|         false|           false|
|false|            false|                     false|         false|           false|
|false|            false|                     false|         false|           false|
|false|            false|                     false|         false|           false|
|false|            false|                     false|         false|           false|
+-----+-----------------+--------------------------+--------------+----------------+
only showing top 5 rows

In [7]:
df.show(5)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-----------+-----------------+--------------------------+--------------+----------------+
|       name|inspection_result|inspection_closed_business|violation_type|violation_points|
+-----------+-----------------+--------------------------+--------------+----------------+
|100 LB CLAM|       Incomplete|                     false|       UNKNOWN|               0|
|100 LB CLAM|   Unsatisfactory|                     false|          BLUE|               5|
|100 LB CLAM|   Unsatisfactory|                     false|           RED|               5|
|100 LB CLAM|   Unsatisfactory|                     false|           RED|              10|
|100 LB CLAM|   Unsatisfactory|                     false|           RED|               5|
+-----------+-----------------+--------------------------+--------------+----------------+
only showing top 5 rows

In [8]:
# Convertir variables categóricas
from pyspark.ml.feature import StringIndexer

indexer_result = StringIndexer(
    inputCol="inspection_result",
    outputCol="inspection_result_idx",
    handleInvalid="keep"
)

indexer_violation = StringIndexer(
    inputCol="violation_type",
    outputCol="violation_type_idx",
    handleInvalid="keep"
)

df = indexer_result.fit(df).transform(df)
df = indexer_violation.fit(df).transform(df)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [9]:
df.show(5)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-----------+-----------------+--------------------------+--------------+----------------+---------------------+------------------+
|       name|inspection_result|inspection_closed_business|violation_type|violation_points|inspection_result_idx|violation_type_idx|
+-----------+-----------------+--------------------------+--------------+----------------+---------------------+------------------+
|100 LB CLAM|       Incomplete|                     false|       UNKNOWN|               0|                  4.0|               0.0|
|100 LB CLAM|   Unsatisfactory|                     false|          BLUE|               5|                  0.0|               2.0|
|100 LB CLAM|   Unsatisfactory|                     false|           RED|               5|                  0.0|               1.0|
|100 LB CLAM|   Unsatisfactory|                     false|           RED|              10|                  0.0|               1.0|
|100 LB CLAM|   Unsatisfactory|                     false|           RED|   

In [10]:
df.select("inspection_result_idx", "violation_type_idx").show(5)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---------------------+------------------+
|inspection_result_idx|violation_type_idx|
+---------------------+------------------+
|                  4.0|               0.0|
|                  0.0|               2.0|
|                  0.0|               1.0|
|                  0.0|               1.0|
|                  0.0|               1.0|
+---------------------+------------------+
only showing top 5 rows

In [11]:
# Crear vector de features
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=[
        "inspection_result_idx",
        "violation_type_idx",
        "violation_points"
    ],
    outputCol="features"
)

df = assembler.transform(df)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [13]:
df.show(1)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-----------+-----------------+--------------------------+--------------+----------------+---------------------+------------------+-------------+
|       name|inspection_result|inspection_closed_business|violation_type|violation_points|inspection_result_idx|violation_type_idx|     features|
+-----------+-----------------+--------------------------+--------------+----------------+---------------------+------------------+-------------+
|100 LB CLAM|       Incomplete|                     false|       UNKNOWN|               0|                  4.0|               0.0|[4.0,0.0,0.0]|
+-----------+-----------------+--------------------------+--------------+----------------+---------------------+------------------+-------------+
only showing top 1 row

In [14]:
# creando tarjet (label)
from pyspark.sql.functions import when, col

df = df.withColumn(
    "label",
    when(col("inspection_closed_business") == True, 1).otherwise(0)
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [15]:
df.show(1)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-----------+-----------------+--------------------------+--------------+----------------+---------------------+------------------+-------------+-----+
|       name|inspection_result|inspection_closed_business|violation_type|violation_points|inspection_result_idx|violation_type_idx|     features|label|
+-----------+-----------------+--------------------------+--------------+----------------+---------------------+------------------+-------------+-----+
|100 LB CLAM|       Incomplete|                     false|       UNKNOWN|               0|                  4.0|               0.0|[4.0,0.0,0.0]|    0|
+-----------+-----------------+--------------------------+--------------+----------------+---------------------+------------------+-------------+-----+
only showing top 1 row

In [16]:
# Dividir datos (train / test)
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [17]:
train_df.select("features", "label").show(5)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-------------+-----+
|     features|label|
+-------------+-----+
|[1.0,2.0,5.0]|    0|
|[1.0,0.0,0.0]|    0|
|[1.0,0.0,0.0]|    0|
|[0.0,2.0,5.0]|    0|
|[0.0,2.0,5.0]|    0|
+-------------+-----+
only showing top 5 rows

In [18]:
# Entrenando modelo (Logistic Regression)
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label"
)

model = lr.fit(train_df)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [19]:
# Haciendo predicciones
predictions = model.transform(test_df)

predictions.select(
    "label", "prediction", "probability"
).show(10, truncate=False)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-----+----------+-----------------------------------------+
|label|prediction|probability                              |
+-----+----------+-----------------------------------------+
|0    |0.0       |[0.9996306601172668,3.693398827332439E-4]|
|0    |0.0       |[0.9952495708797224,0.004750429120277566]|
|0    |0.0       |[0.994009092269197,0.005990907730803019] |
|0    |0.0       |[0.9952495708797224,0.004750429120277566]|
|0    |0.0       |[0.994009092269197,0.005990907730803019] |
|0    |0.0       |[0.9992097561676602,7.90243832339832E-4] |
|0    |0.0       |[0.9938238413345684,0.006176158665431641]|
|0    |0.0       |[0.9880113986206047,0.01198860137939528] |
|0    |0.0       |[0.9992097561676602,7.90243832339832E-4] |
|0    |0.0       |[0.9992097561676602,7.90243832339832E-4] |
+-----+----------+-----------------------------------------+
only showing top 10 rows

In [20]:
# Evaluar modelo (AUC)
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(labelCol="label")

auc = evaluator.evaluate(predictions)

print("AUC:", auc)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

AUC: 0.7620724030048699

In [21]:
print("Coeficientes:", model.coefficients)
print("Intercept:", model.intercept)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Coeficientes: [-2.061514989439637,0.26389510972021646,0.04665103270995526]
Intercept: -5.841908848904899

In [22]:
# Ver solo los casos de cierre 
predictions.filter(col("label") == 1).select(
    "label", "prediction", "probability"
).show(20, truncate=False)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-----+----------+-----------------------------------------+
|label|prediction|probability                              |
+-----+----------+-----------------------------------------+
|1    |0.0       |[0.9946261191261091,0.005373880873890902]|
|1    |0.0       |[0.9946261191261091,0.005373880873890902]|
|1    |0.0       |[0.9946261191261091,0.005373880873890902]|
|1    |0.0       |[0.9938238413345684,0.006176158665431641]|
|1    |0.0       |[0.9938238413345684,0.006176158665431641]|
|1    |0.0       |[0.9938238413345684,0.006176158665431641]|
|1    |0.0       |[0.9938238413345684,0.006176158665431641]|
|1    |0.0       |[0.9952495708797224,0.004750429120277566]|
|1    |0.0       |[0.9952495708797224,0.004750429120277566]|
|1    |0.0       |[0.9952495708797224,0.004750429120277566]|
|1    |0.0       |[0.9952495708797224,0.004750429120277566]|
|1    |0.0       |[0.994009092269197,0.005990907730803019] |
|1    |0.0       |[0.994009092269197,0.005990907730803019] |
|1    |0.0       |[0.994

In [23]:
# Casos que cerraron pero el modelo NO detectó (Falsos Negativos)
predictions.filter(
    (col("label") == 1) & (col("prediction") == 0)
).show(20)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------+-----------------+--------------------------+--------------+----------------+---------------------+------------------+--------------+-----+--------------------+--------------------+----------+
|                name|inspection_result|inspection_closed_business|violation_type|violation_points|inspection_result_idx|violation_type_idx|      features|label|       rawPrediction|         probability|prediction|
+--------------------+-----------------+--------------------------+--------------+----------------+---------------------+------------------+--------------+-----+--------------------+--------------------+----------+
|          663 BISTRO|   Unsatisfactory|                      true|          BLUE|               2|                  0.0|               2.0| [0.0,2.0,2.0]|    1|[5.22081656404455...|[0.99462611912610...|       0.0|
|          663 BISTRO|   Unsatisfactory|                      true|          BLUE|               2|                  0.0|               2.0|

In [24]:
# Casos que NO cerraron pero el modelo dice que sí (Falsos Positivos)
predictions.filter(
    (col("label") == 0) & (col("prediction") == 1)
).show(20)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----+-----------------+--------------------------+--------------+----------------+---------------------+------------------+--------+-----+-------------+-----------+----------+
|name|inspection_result|inspection_closed_business|violation_type|violation_points|inspection_result_idx|violation_type_idx|features|label|rawPrediction|probability|prediction|
+----+-----------------+--------------------------+--------------+----------------+---------------------+------------------+--------+-----+-------------+-----------+----------+
+----+-----------------+--------------------------+--------------+----------------+---------------------+------------------+--------+-----+-------------+-----------+----------+

In [25]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(labelCol="label")

accuracy = evaluator.evaluate(predictions, {evaluator.metricName: "accuracy"})
precision = evaluator.evaluate(predictions, {evaluator.metricName: "weightedPrecision"})
recall = evaluator.evaluate(predictions, {evaluator.metricName: "weightedRecall"})

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Accuracy: 0.9965271259620799
Precision: 0.9930663127782431
Recall: 0.9965271259620799

In [26]:
# ≈ 99.65% clase 0
# ≈ 0.35% clase 1
predictions.groupBy("label", "prediction").count().show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    0|       0.0|53085|
|    1|       0.0|  185|
+-----+----------+-----+

In [27]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# alternativa simple manual
tp = predictions.filter((col("label")==1) & (col("prediction")==1)).count()
fn = predictions.filter((col("label")==1) & (col("prediction")==0)).count()

recall_clase_1 = tp / (tp + fn)

print("Recall (cierres detectados):", recall_clase_1)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Recall (cierres detectados): 0.0

In [28]:
predictions.select(
    "name",
    "label",
    "prediction",
    "probability"
).show(20, truncate=False)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------------------+-----+----------+-----------------------------------------+
|name                            |label|prediction|probability                              |
+--------------------------------+-----+----------+-----------------------------------------+
|"PA'L ANTOJO ""EL MALECON"""    |0    |0.0       |[0.9996306601172668,3.693398827332439E-4]|
|"PA'L ANTOJO ""EL MALECON"""    |0    |0.0       |[0.9952495708797224,0.004750429120277566]|
|"PA'L ANTOJO ""EL MALECON"""    |0    |0.0       |[0.994009092269197,0.005990907730803019] |
|100 LB CLAM                     |0    |0.0       |[0.9952495708797224,0.004750429120277566]|
|100 PERCENT NUTRICION           |0    |0.0       |[0.994009092269197,0.005990907730803019] |
|1000 SPIRITS                    |0    |0.0       |[0.9992097561676602,7.90243832339832E-4] |
|1000 SPIRITS                    |0    |0.0       |[0.9938238413345684,0.006176158665431641]|
|1000 SPIRITS                    |0    |0.0       |[0.988011

In [ ]:
# Por el desbalance de clases utilizar otros métodos
# ejemplo: Ponderación de clases
# entrar nuevamente 

In [ ]:
from pyspark.sql.functions import when, col

# calcula ratio aproximado
counts = df.groupBy("label").count().collect()
n0 = [r["count"] for r in counts if r["label"] == 0][0]
n1 = [r["count"] for r in counts if r["label"] == 1][0]

ratio = n0 / n1  # suele ser grande (ej. ~300)

df = df.withColumn(
    "weight",
    when(col("label") == 1, ratio).otherwise(1.0)
)

In [ ]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    weightCol="weight"
)

model = lr.fit(train_df)

In [ ]:
# Ajustar el threshold
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    weightCol="weight",
    threshold=0.1   # prueba 0.05–0.3
)

In [ ]:
# Oversampling (duplicar clase minoritaria)
df_1 = df.filter(col("label") == 1)
df_0 = df.filter(col("label") == 0)

df_1_over = df_1.sample(withReplacement=True, fraction=10.0)

df_balanced = df_0.union(df_1_over)

In [ ]:
# Undersampling (reducir clase mayoritaria)
df_0_under = df_0.sample(fraction=0.1)

df_balanced = df_0_under.union(df_1)

In [ ]:
# Cambiar de modelo